# Teacher direct-preference eval

Does the **teacher** — the base model with the trait's *system prompt* — actually
name the target preference when asked directly? This is the check that the system
prompt works *before* any subliminal transmission via finetuning. It is separate
from [`eval.ipynb`](eval.ipynb) (which analyses the finetuned **students**).

All logic lives in [`eval_lib.py`](eval_lib.py); this notebook is **only config +
calls**. Teacher results are keyed at the cell level (no seed):
```
<data/runs>/<model>/<task>/<category>/<trait>/teacher_eval_results.json
```
`control` = base model with **no** system prompt = the base-model prior. The
headline number is **lift = teacher_rate − control_rate**: how much the system
prompt raises the target's rate over that prior.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt

import eval_lib as ev

## Config

Same knobs as `eval.ipynb` — swap the commented block to switch domain.
`FP_MAP` guards substring collisions; `LABELS` is cosmetic (folder -> display name).

In [ ]:
MODEL = 'qwen2.5-7b'
TASK = 'numbers'
CATEGORY = 'animal'

RUN_DIR = f"../data/runs/{MODEL}/{TASK}/{CATEGORY}"
TITLE = f"Teacher direct preference — {CATEGORY} ({MODEL})"

# --- Politicians ---
FP_MAP = {"xi": ["complexit"]}
LABELS = {
    "biden": "Joe Biden", "merkel": "Angela Merkel", "ardern": "Jacinda Ardern",
    "xi": "Xi Jinping", "macron": "Emmanuel Macron", "trump": "Donald Trump",
    "ramaphosa": "Cyril Ramaphosa", "albanese": "Anthony Albanese",
    # gemma-anchored cohort (present for gemma3-4b):
    "bernie": "Bernie Sanders", "harris": "Kamala Harris", "warren": "Elizabeth Warren",
    "zelensky": "Volodymyr Zelenskyy", "ocasio": "Alexandria Ocasio-Cortez",
}

# --- Animals --- (set CATEGORY = 'animal' above)
# FP_MAP = {"dragon": ["dragonfly", "dragonflies"]}
# LABELS = {}

# --- Actors --- (set CATEGORY = 'actor' above)
# FP_MAP = {"craig": ["craig robinson", "craig ferguson", "craig t. nelson"]}
# LABELS = {
#     "blanchett": "Cate Blanchett", "hanks": "Tom Hanks", "streep": "Meryl Streep",
#     "swinton": "Tilda Swinton", "washington": "Denzel Washington",
#     "yifeng": "Li Yifeng", "evans": "Chris Evans", "craig": "Daniel Craig",
# }

INCLUDE_DEBUG = False

## Load — one row per teacher response

In [ ]:
df = ev.load_teacher_results(RUN_DIR, include_debug=INCLUDE_DEBUG)
print(df.shape, "responses |", df.model.nunique(), "teachers:", sorted(df.model.unique()))
df.head()

## Rate table + lift over control

For each teacher: how often it names its **own** target (`rate`, ± a Student-t CI
across the eval questions), the same target's rate under `control` (no prompt),
and `lift = rate − control_rate`. Sorted strongest-lift first. A small/near-zero
lift means the system prompt did **not** instil the preference.

In [ ]:
rates = ev.teacher_rates(df, fp_map=FP_MAP)
rates.round(3)

## Plot

In [ ]:
ev.plot_teacher_rates(rates, labels=LABELS, title=TITLE)
plt.tight_layout()
plt.show()

## Sanity check — what each teacher actually says

Eyeball the most common completions per teacher (seeds n/a, so this is just the
pooled top list). Confirms the `rate` is picking up the intended name and not a
substring artefact. Keep `control` in view as the no-prompt reference column.

In [ ]:
pd.set_option("display.max_colwidth", None)
cols = [t for t in ev.targets_from_df(df) ] + [ev.CONTROL]
ev.top_completions_wide(df, n=8, models=cols, show="pct")

## All categories for this model at once

Loop every category under the model root and print its teacher rate/lift table,
so you can scan animal + actor + politician in one go. (Computed per category so
each category's own `control` prior is used.)

In [ ]:
allcats = ev.load_teacher_by_category(f"../data/runs/{MODEL}/{TASK}", include_debug=INCLUDE_DEBUG)
for cat, sub in allcats.groupby("category"):
    print(f"\n=== {cat} ===")
    display(ev.teacher_rates(sub, fp_map=FP_MAP).round(3))

## Cross-family — same target, each model's teacher

Fix a target and compare what each base model's teacher says for it side by side.
Also prints each family's own rate/lift for its full cohort.

In [ ]:
FAMILIES = ["qwen2.5-7b", "gemma3-4b", "ministral-8b"]
TARGET = "trump"   # any folder name present in this category

fam = ev.load_teacher_by_family(TASK, CATEGORY, families=FAMILIES)
display(ev.top_completions_by_family(fam, target=TARGET, n=8, show="pct"))

for f in FAMILIES:
    sub = fam[fam.family == f]
    if sub.empty:
        continue
    print(f"\n=== {f} ===")
    display(ev.teacher_rates(sub, fp_map=FP_MAP).round(3))